# Notebook 01: Read & Inspect Gold Layer Documents
This notebook reads the Gold Layer Parquet datasets (`business_documents` and `review_documents`) locally, inspecting document schemas, record counts, and document text structures.

In [1]:
import os
import sys
import duckdb
import pandas as pd
from pathlib import Path
from IPython.display import display

# Setup paths relative to project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

GOLD_BIZ_PATH = PROJECT_ROOT / 'data' / '02_gold' / 'business_documents'
GOLD_REV_PATH = PROJECT_ROOT / 'data' / '02_gold' / 'review_documents'

biz_glob = str(GOLD_BIZ_PATH / '*.parquet').replace('\\', '/')
rev_glob = str(GOLD_REV_PATH / '*.parquet').replace('\\', '/')

# Initialize DuckDB connection
con = duckdb.connect()

print(f'Project Root:  {PROJECT_ROOT}')
print(f'Business Path: {GOLD_BIZ_PATH}')
print(f'Review Path:   {GOLD_REV_PATH}')


Project Root:  C:\Users\shank\Downloads\yelp_project
Business Path: C:\Users\shank\Downloads\yelp_project\data\02_gold\business_documents
Review Path:   C:\Users\shank\Downloads\yelp_project\data\02_gold\review_documents


In [2]:
print("--> Business Documents Schema & Sample Rows:")
biz_df = con.execute(f"""
    SELECT 
        document_id, business_name, city, state, 
        primary_category, business_rating, review_count, price_range, is_open
    FROM read_parquet('{biz_glob}')
    LIMIT 5
""").df()

display(biz_df)


--> Business Documents Schema & Sample Rows:


,document_id,business_name,city,state,primary_category,business_rating,review_count,price_range,is_open
0,doc_biz_--30_8IhuyMHbSOcNWd6DQ,Action Karate,Jamison,PA,Trainers,3.5,9,NaN,1
1,doc_biz_--FcbSxK1AoEtEAxOgBaCw,Victory Car Wash,Riverview,FL,Car Wash,3.5,40,NaN,1
2,doc_biz_--SJXpAa0E-GCp2smaHf0A,Winn Dixie,Riverview,FL,Grocery,2.5,13,2,1
3,doc_biz_--ZVrH2X2QXBFdCilbirsw,Chris's Sandwich Shop,Ardmore,PA,American (Traditional),4.5,32,1,0
4,doc_biz_--a_r_w1HTsOY-fagPeNKg,Binford Farmers Market,Indianapolis,IN,Farmers Market,4.0,5,3,0


In [3]:
print("--> Sample Business Document Text:")
sample_biz_text = con.execute(f"SELECT document_text FROM read_parquet('{biz_glob}') LIMIT 1").fetchone()[0]
print("------------------ DOCUMENT TEXT ------------------")
print(sample_biz_text)
print("---------------------------------------------------")


--> Sample Business Document Text:
------------------ DOCUMENT TEXT ------------------
Business Name: Action Karate
Primary Category: Trainers
Categories: Trainers, Active Life, Fitness & Instruction, Karate, Martial Arts
Location: 2235 York Rd, Jamison, PA 18929
Coordinates: Lat 40.2553619, Lon -75.0883992
Rating: 3.5 stars (9 reviews)
Price Range: N/A
Operating Status: Open
Hours: 
Features: creditcards=True, 
---------------------------------------------------


In [4]:
print("--> Review Documents Schema & Sample Rows:")
rev_df = con.execute(f"""
    SELECT 
        document_id, review_id, business_name, city, state, 
        stars, sentiment, review_date
    FROM read_parquet('{rev_glob}')
    LIMIT 5
""").df()

display(rev_df)


--> Review Documents Schema & Sample Rows:


,document_id,review_id,business_name,city,state,stars,sentiment,review_date
0,doc_rev_LJlThM2hOOKBWZAj9YqWVw,LJlThM2hOOKBWZAj9YqWVw,Homewood Suites by Hilton New Orleans French Q...,New Orleans,LA,2.0,negative,2018-05-30
1,doc_rev_LJlUzK-5RniNx7lm88AoTw,LJlUzK-5RniNx7lm88AoTw,Popeyes Louisiana Kitchen,Wesley Chapel,FL,1.0,negative,2021-03-20
2,doc_rev_LJl_stLAdy-0ETC0Pcm17w,LJl_stLAdy-0ETC0Pcm17w,Molly Maguire's Irish Restaurant And Pub,Phoenixville,PA,3.0,neutral,2012-06-23
3,doc_rev_LJlen_gDOedc_fanRySlMg,LJlen_gDOedc_fanRySlMg,The Coffee House at Second and Bridge,Franklin,TN,4.0,positive,2019-09-11
4,doc_rev_LJlk6gRJrkevHTFK8w2ptw,LJlk6gRJrkevHTFK8w2ptw,Wig Villa,Saint Petersburg,FL,1.0,negative,2020-01-25


In [5]:
print("--> Sample Review Document Text:")
sample_rev_text = con.execute(f"SELECT document_text FROM read_parquet('{rev_glob}') LIMIT 1").fetchone()[0]
print("------------------ DOCUMENT TEXT ------------------")
print(sample_rev_text)
print("---------------------------------------------------")
con.close()


--> Sample Review Document Text:


------------------ DOCUMENT TEXT ------------------
Review for Homewood Suites by Hilton New Orleans French Quarter (New Orleans, LA)
Category: Hotels
Review Rating: 2.0 stars
Date: 2018-05-30
Sentiment: negative
Content: I love this place in the beginning. We stayed tor 8 days. Hot breakfast 7 days a week and happy hour M-Thursday included    What's not to like. However, as the week, progressed there were definite problems. 
The breakfast deteriorated.  There were always empty stations.  No coffee, no eggs, no bread products and they ran out of CATSUP today. 
The room... the shower was clogged and in order to turn it on you got wet because the door opens on the far side and you have to step in to turn it on.  
The sheets on the bed were too small so every night the sheets became a tangled mess because they came undone. Also, the sheets were never changed in in fact some days we had to call for maid service at 4:00 because the room hasn't been done. 

The check in staff was great until